In [ ]:
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

if "google.colab" in sys.modules:
    archive_path = Path("/content/snake_dqn.zip")
    repository_path = Path("/content/snake_dqn")
    urlretrieve(
        "https://github.com/Mazako/snake_dqn/archive/refs/heads/master.zip",
        archive_path,
    )
    with ZipFile(archive_path) as archive:
        archive.extractall(repository_path)

    repository_root = next(path for path in repository_path.iterdir() if path.is_dir())
    os.chdir(repository_root)
    sys.path.insert(0, str(repository_root))

In [ ]:
import numpy as np
import torch
from torch import nn


In [ ]:
import copy
from random import Random

from snake_dqn.agent import RelativeAction
from snake_dqn.game import Game
from snake_dqn.replay_buffer import ReplayBuffer, Transition


In [ ]:
device_type = "cpu"
if torch.cuda.is_available():
    device_type = "cuda"
elif torch.mps.is_available():
    device_type = "mps"

device = torch.device(device_type)
print(f"{device=}")


device=device(type='mps')


In [ ]:

seed = 42
rng = Random(seed)
torch.manual_seed(seed)

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1, padding_mode="circular"),
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1, padding_mode="circular"),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(32 * 11 * 11, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
).to(device)

checkpoint_path = Path("best_model.pt")
best_eval_score = float("-inf")
if checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_eval_score = float(checkpoint["best_eval_score"])
    print(f"Loaded model with best eval score={best_eval_score:.2f}")

target_model = copy.deepcopy(model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.SmoothL1Loss()
buffer = ReplayBuffer(100_000, rng)

In [ ]:
def select_actions(
    model: nn.Module,
    states: torch.Tensor,
    epsilon: float,
    rng: Random,
    device: torch.device,
) -> list[RelativeAction]:
    if epsilon == 1.0:
        return [rng.choice(tuple(RelativeAction)) for _ in states]

    with torch.no_grad():
        action_indices = model(states.to(device)).argmax(dim=1).tolist()

    return [
        rng.choice(tuple(RelativeAction))
        if rng.random() < epsilon
        else RelativeAction(action_index)
        for action_index in action_indices
    ]


def train_step(
    model: nn.Module,
    target_model: nn.Module,
    buffer: ReplayBuffer,
    batch_size: int,
    gamma: float,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float | None:
    if len(buffer) < batch_size:
        return None

    states, actions_batch, rewards, next_states, dones = (
        tensor.to(device) for tensor in buffer.sample(batch_size)
    )

    chosen_q_values = model(states).gather(1, actions_batch.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_actions = model(next_states).argmax(dim=1, keepdim=True)
        next_q_values = target_model(next_states).gather(1, next_actions).squeeze(1)
        targets = rewards + gamma * (1 - dones) * next_q_values

    loss = loss_fn(chosen_q_values, targets)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
from collections import deque


def evaluate(
    model: nn.Module,
    episodes: int,
    board_size: int,
    max_steps: int,
    device: torch.device,
) -> float:
    was_training = model.training
    model.eval()
    scores = []

    try:
        with torch.no_grad():
            for _ in range(episodes):
                game = Game(board_size, max_steps)
                state = game.state_img()

                while True:
                    action_index = model(state.to(device).unsqueeze(0)).argmax(dim=1).item()
                    _, _, done = game.step(RelativeAction(action_index))
                    state = game.state_img()

                    if done:
                        break

                scores.append(game.score)
    finally:
        model.train(was_training)

    return float(np.mean(scores))


def train(
    model: nn.Module,
    target_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    buffer: ReplayBuffer,
    rng: Random,
    device: torch.device,
    episodes: int,
    board_size: int,
    max_steps: int,
    gamma: float,
    batch_size: int,
    epsilon: float,
    epsilon_min: float,
    epsilon_decay: float,
    num_envs: int,
    updates_per_batch: int,
    target_sync_interval: int,
    log_interval_iterations: int,
    eval_interval_iterations: int,
    eval_episodes: int,
    best_eval_score: float,
    checkpoint_path: Path,
) -> tuple[list[int], list[float], list[tuple[int, float]]]:
    environment_steps = 0
    next_target_sync_step = target_sync_interval
    completed_episodes = 0
    iteration = 0
    scores = []
    losses = []
    evaluations = []
    scores_avg = deque(maxlen=100)
    games = [Game(board_size, max_steps) for _ in range(num_envs)]
    states = [game.state_img() for game in games]

    while completed_episodes < episodes:
        actions = select_actions(model, torch.stack(states), epsilon, rng, device)

        for index, (game, state, action) in enumerate(zip(games, states, actions)):
            _, reward, done = game.step(action)
            next_state = game.state_img()

            buffer.append(Transition(state, action, reward, next_state, done))
            environment_steps += 1

            if done:
                completed_episodes += 1
                scores.append(game.score)
                scores_avg.append(game.score)
                epsilon = max(epsilon_min, epsilon * epsilon_decay)
                games[index] = Game(board_size, max_steps)
                states[index] = games[index].state_img()
            else:
                states[index] = next_state

            if completed_episodes == episodes:
                break

        for _ in range(updates_per_batch):
            loss = train_step(
                model,
                target_model,
                buffer,
                batch_size,
                gamma,
                loss_fn,
                optimizer,
                device,
            )
            if loss is None:
                break
            losses.append(loss)

        iteration += 1

        if environment_steps >= next_target_sync_step:
            target_model.load_state_dict(model.state_dict())
            next_target_sync_step += target_sync_interval

        if iteration % log_interval_iterations == 0:
            mean_loss = np.mean(losses[-100:]) if losses else float("nan")
            mean_score = np.mean(scores_avg) if scores_avg else float("nan")
            last_score = scores[-1] if scores else "n/a"
            print(
                f"iteration={iteration} episodes={completed_episodes} "
                f"steps={environment_steps} score={last_score} "
                f"epsilon={epsilon:.3f} loss={mean_loss:.4f} "
                f"running avg={mean_score:.2f}"
            )

        if iteration % eval_interval_iterations == 0 or completed_episodes == episodes:
            eval_score = evaluate(
                model,
                eval_episodes,
                board_size,
                max_steps,
                device,
            )
            evaluations.append((iteration, eval_score))
            print(
                f"evaluation iteration={iteration} episodes={completed_episodes} "
                f"avg score={eval_score:.2f}"
            )
            if eval_score > best_eval_score:
                best_eval_score = eval_score
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "best_eval_score": best_eval_score,
                    },
                    checkpoint_path,
                )
                print(f"Saved best model with eval score={best_eval_score:.2f}")

    return scores, losses, evaluations


scores, losses, evaluations = train(
    model=model,
    target_model=target_model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    buffer=buffer,
    rng=rng,
    device=device,
    episodes=5_000,
    board_size=11,
    max_steps=200,
    gamma=0.99,
    batch_size=128,
    epsilon=0.2 if checkpoint_path.is_file() else 0.95,
    epsilon_min=0.01,
    epsilon_decay=0.9994,
    num_envs=256,
    updates_per_batch=32,
    target_sync_interval=10_000,
    log_interval_iterations=10,
    eval_interval_iterations=200,
    eval_episodes=100,
    best_eval_score=best_eval_score,
    checkpoint_path=checkpoint_path,
)

iteration=10 episodes=0 steps=2560 score=n/a epsilon=0.950 loss=0.0017 running avg=nan
iteration=20 episodes=0 steps=5120 score=n/a epsilon=0.950 loss=0.0012 running avg=nan
iteration=30 episodes=0 steps=7680 score=n/a epsilon=0.950 loss=0.0008 running avg=nan
iteration=40 episodes=0 steps=10240 score=n/a epsilon=0.950 loss=0.0007 running avg=nan
iteration=50 episodes=0 steps=12800 score=n/a epsilon=0.950 loss=0.0007 running avg=nan
iteration=60 episodes=0 steps=15360 score=n/a epsilon=0.950 loss=0.0018 running avg=nan
iteration=70 episodes=0 steps=17920 score=n/a epsilon=0.950 loss=0.0013 running avg=nan
iteration=80 episodes=0 steps=20480 score=n/a epsilon=0.950 loss=0.0310 running avg=nan
iteration=90 episodes=0 steps=23040 score=n/a epsilon=0.950 loss=0.0214 running avg=nan
iteration=100 episodes=0 steps=25600 score=n/a epsilon=0.950 loss=0.0213 running avg=nan
iteration=110 episodes=1 steps=28160 score=4 epsilon=0.949 loss=0.0240 running avg=4.00
iteration=120 episodes=1 steps=307

KeyboardInterrupt: 